## Self Attention mechanism

In [1]:
import torch

# Your Journey starts with one step
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], #x^1
    [0.55, 0.87, 0.66], #x^2
    [0.57, 0.85, 0.64], #x^3
    [0.22, 0.58, 0.33], #x^4
    [0.77, 0.25, 0.10], #x^5
    [0.05, 0.80, 0.55]] #x^6
)

In [8]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2
print(f"Inputs {x_2} with dimensions of {d_in} and output {d_out}")

Inputs tensor([0.5500, 0.8700, 0.6600]) with dimensions of 3 and output 2


In [13]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)
W_key= torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)


In [14]:
print(W_query, W_key, W_value)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]]) Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]]) Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


In [15]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

In [20]:
print(query_2, query_2.shape)

tensor([0.4306, 1.4551]) torch.Size([2])


In [18]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape",keys.shape)
print("values shape", values.shape)

keys.shape torch.Size([6, 2])
values shape torch.Size([6, 2])


In [21]:
print(keys,values)

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]]) tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])


## attention weights

In [22]:
keys_2 = keys[1]
attention_score_22 = query_2.dot(keys_2)
print(attention_score_22)

tensor(1.8524)


In [24]:
attention_scores_2 = query_2 @ keys.T
print(attention_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [26]:
d_k = keys.shape[-1]
attention_weight_2 = torch.softmax(attention_scores_2/d_k**0.5, dim =-1)
print(attention_weight_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [27]:
all_context_vectors = attention_weight_2 @ values
print(all_context_vectors)

tensor([0.3061, 0.8210])


## A compact self attention class

In [ ]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
    '''A simple implementation of self attention mechanism
    
    Args:
        d_in: input dimension
        d_out: output dimension
    
    '''

    def __init__(self, d_in ,d_out ):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self,x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim =-1)
        context_vector = attention_weights @ values
        return context_vector


In [38]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in,d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


## Self attention using pytorch Linear layers

In [ ]:
class SelfAttention_v2(nn.Module):
    '''A simple implementation of self attention mechanism

    Args:
        d_in: input dimension
        d_out: output dimension
        qkv_bias: whether to include bias in the linear layers for query, key, and value projections (default: False)
    
    For each input token, the module computes query, key, and value vectors using separate linear transformations. 
    The attention scores are calculated as the dot product of the query with all keys, scaled by the square root of the key dimension. 
    The attention weights are obtained by applying a softmax function to the attention scores, 
    and the context vector is computed as a weighted sum of the value vectors based on these attention weights.
    '''
    def __init__(self, d_in ,d_out, qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)

    def forward(self,x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim =-1)
        context_vector = attention_weights @ values
        return context_vector

In [45]:
torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in,d_out)
print(sa_v2(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


In [53]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in,d_out)
sa_v2 = SelfAttention_v2(d_in,d_out)
sa_v1.W_query = nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key = nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = nn.Parameter(sa_v2.W_value.weight.T)

print(sa_v1(inputs))
print(sa_v2(inputs))



tensor([[0.5085, 0.3508],
        [0.5084, 0.3508],
        [0.5084, 0.3506],
        [0.5074, 0.3471],
        [0.5076, 0.3446],
        [0.5077, 0.3493]], grad_fn=<MmBackward0>)
tensor([[0.5085, 0.3508],
        [0.5084, 0.3508],
        [0.5084, 0.3506],
        [0.5074, 0.3471],
        [0.5076, 0.3446],
        [0.5077, 0.3493]], grad_fn=<MmBackward0>)
